# 01 · Data preparation
**Lighting the Way? Street light availability and street crime across urban Northern Ireland**

This notebook cleans the three main datasets and puts them on one common scale:
NISRA's Census 2021 **1 km grid squares**.

1. NISRA grid: boundaries and population
2. DfI street lights: cleaning and lamp type
3. data.police.uk crime (PSNI), July 2023 – June 2024
4. OpenStreetMap commercial points of interest (optional)
5. Aggregation to grid cells, the Street Light Availability Index (SLAI) and the urban subset

Download the data first: see `data/README.md`. Check the settings marked **CHECK** in `src/config.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # lets the notebook import from src/

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src import config

config.FIGURES.mkdir(parents=True, exist_ok=True)
config.TABLES.mkdir(parents=True, exist_ok=True)

from src.features import classify_lamp, build_slai
from src.modelling import make_blocks
from src.settlements import settlement_coverage

cleaning_log = []   # row counts at each cleaning step, for the Data and Methods section

def log(dataset, step, n):
    cleaning_log.append({"dataset": dataset, "step": step, "rows": n})
    print(f"{dataset:12s} | {step:45s} | {n:,}")

## 1. NISRA 1 km grid and population

Print the column names first, then set `GRID_ID_COL_BOUNDARY`, `GRID_ID_COL_DATA`
and `POP_COL` in `src/config.py` to match.

In [ ]:
grid = gpd.read_file(config.GRID_BOUNDARY)
pop = pd.read_csv(config.GRID_DATA_CSV)

print("Boundary CRS:", grid.crs)
print("Boundary columns:", grid.columns.tolist())
print("Data columns (first 30):", pop.columns.tolist()[:30])

In [ ]:
grid = grid.rename(columns={config.GRID_ID_COL_BOUNDARY: "cell_id"})
pop = pop.rename(columns={config.GRID_ID_COL_DATA: "cell_id", config.POP_COL: "population"})

grid = grid[["cell_id", "geometry"]].merge(pop[["cell_id", "population"]], on="cell_id", how="left")
grid["population"] = pd.to_numeric(grid["population"], errors="coerce").fillna(0)
grid = grid.to_crs(config.CRS)
grid["area_km2"] = grid.area / 1e6

log("grid", "1 km cells", len(grid))
log("grid", "cells with residents", int((grid.population > 0).sum()))
grid.head()

## 2. Street lights (DfI Roads, OpenDataNI)

The asset register records where lighting columns are installed, not whether
they were working. DfI also notes that some asset locations are inaccurate.
Both points belong in the limitations.

In [ ]:
lights_raw = pd.read_csv(config.STREETLIGHT_CSV, low_memory=False)
log("lights", "raw rows", len(lights_raw))
lights_raw["ITEM_TYPE_NAME"].value_counts()

In [ ]:
lights = lights_raw.copy()
if config.LIGHT_ITEM_TYPES:
    lights = lights[lights["ITEM_TYPE_NAME"].isin(config.LIGHT_ITEM_TYPES)]
    log("lights", f"item type in {config.LIGHT_ITEM_TYPES}", len(lights))

lights["EASTING"] = pd.to_numeric(lights["EASTING"], errors="coerce")
lights["NORTHING"] = pd.to_numeric(lights["NORTHING"], errors="coerce")
in_ni = (lights.EASTING.between(*config.NI_EASTING_RANGE)
         & lights.NORTHING.between(*config.NI_NORTHING_RANGE))
lights = lights[in_ni]
log("lights", "valid Irish Grid coordinates", len(lights))

# Several lanterns can share one column location; count columns, not lanterns
lights = lights.drop_duplicates(subset=["EASTING", "NORTHING"])
log("lights", "duplicate locations removed", len(lights))

lights["lamp_type"] = classify_lamp(lights["LANTERN_BOQ_NAME"])
lights["mount_height"] = pd.to_numeric(lights["MOUNT_HEIGHT"], errors="coerce")
lights.loc[~lights.mount_height.between(*config.MOUNT_HEIGHT_RANGE), "mount_height"] = np.nan
lights["lamp_type"].value_counts(normalize=True).round(3)

lights = gpd.GeoDataFrame(
    lights, geometry=gpd.points_from_xy(lights.EASTING, lights.NORTHING),
    crs=config.STREETLIGHT_CRS,
).to_crs(config.CRS)

## 3. Crime (data.police.uk, PSNI)

police.uk locations are snapped to anonymised points on the street network,
and there is no time of day. Anti-social behaviour is removed because it is
recorded as incidents, not crimes.

In [ ]:
files = sorted(config.POLICE_DIR.rglob("*-street.csv"))
print(f"{len(files)} monthly files found")
crime_raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
log("crime", "raw rows", len(crime_raw))

crime = crime_raw[crime_raw["Month"].between(config.START_MONTH, config.END_MONTH)].copy()
log("crime", f"months {config.START_MONTH} to {config.END_MONTH}", len(crime))

crime = crime[crime["Crime type"] != "Anti-social behaviour"]
log("crime", "anti-social behaviour removed", len(crime))

crime = crime.dropna(subset=["Longitude", "Latitude"])
log("crime", "valid coordinates", len(crime))

# PSNI reuses Crime IDs across months for different crimes, so an ID is only
# unique within a month: check duplicates on (Crime ID, Month) together.
crime = crime.drop_duplicates(subset=["Crime ID", "Month"])
log("crime", "duplicate (Crime ID, Month) removed", len(crime))

crime["group"] = "other"
for group, types in config.CRIME_GROUPS.items():
    crime.loc[crime["Crime type"].isin(types), "group"] = group

crime = gpd.GeoDataFrame(
    crime, geometry=gpd.points_from_xy(crime.Longitude, crime.Latitude), crs=config.POLICE_CRS,
).to_crs(config.CRS)

crime.groupby(["group", "Crime type"]).size().rename("count").to_frame()

## 4. Commercial activity (OpenStreetMap, optional)

In [ ]:
if config.OSM_POIS.exists():
    pois = gpd.read_file(config.OSM_POIS)
    pois = pois[pois["fclass"].isin(config.COMMERCIAL_FCLASSES)].to_crs(config.CRS)
    log("pois", "commercial points of interest", len(pois))
else:
    pois = None
    print("No OSM file found: commercial activity will be left out.")

## 5. Aggregate to grid cells

Each point is assigned to the 1 km cell that contains it. Points outside the
grid (e.g. offshore or mislocated) are dropped by the spatial join.

In [ ]:
def assign_cells(points):
    joined = gpd.sjoin(points, grid[["cell_id", "geometry"]], predicate="within", how="inner")
    return joined.drop(columns="index_right")

lights_c = assign_cells(lights)
crime_c = assign_cells(crime)
log("lights", "inside the grid", len(lights_c))
log("crime", "inside the grid", len(crime_c))

cells = grid.set_index("cell_id")
cells["n_lights"] = lights_c.groupby("cell_id").size()
cells["pct_led"] = lights_c.assign(led=lights_c.lamp_type == "LED").groupby("cell_id")["led"].mean() * 100
cells["mean_mount_height"] = lights_c.groupby("cell_id")["mount_height"].mean()

for group in list(config.CRIME_GROUPS) + ["other"]:
    cells[f"{group}_count"] = crime_c[crime_c.group == group].groupby("cell_id").size()

# counts per individual crime type, for extra checks
type_counts = crime_c.groupby(["cell_id", "Crime type"]).size().unstack(fill_value=0)
type_counts.columns = ["n_" + c.lower().replace(" ", "_") for c in type_counts.columns]
cells = cells.join(type_counts)

if pois is not None:
    cells["n_poi"] = assign_cells(pois).groupby("cell_id").size()
else:
    cells["n_poi"] = np.nan

count_cols = [c for c in cells.columns if c.startswith("n_") or c.endswith("_count")]
cells[count_cols] = cells[count_cols].fillna(0)

### Derived variables and the Street Light Availability Index (SLAI)

The SLAI averages two components, each rescaled to 0–1 using **percentile ranks**
(robust to extreme city-centre values; unlit cells score 0):

- lights per km² (how dense the lighting is)
- lights per 1,000 residents (lighting relative to the people living there)

Lantern wattage was considered but is recorded for only ~12% of lanterns, almost
all older sodium lamps. Lamp type (% LED) is kept as a separate descriptive variable.

**Study area.** Urban squares are those at least 50% inside a NISRA Settlement Development
Limit (2015) of Band A–E (settlements of 5,000+ people, NISRA's default urban threshold).
Squares with fewer than 100 residents (harbours, industrial estates, parks) are excluded
because per-resident rates there are unstable. The SLAI is built within this study area.

In [ ]:
cells["pop_density"] = cells.population / cells.area_km2
cells["lights_per_km2"] = cells.n_lights / cells.area_km2
cells["lights_per_1000"] = np.where(cells.population > 0, cells.n_lights / cells.population * 1000, np.nan)
cells["lit"] = (cells.n_lights > 0).astype(int)

for group in config.CRIME_GROUPS:
    cells[f"{group}_rate"] = np.where(
        cells.population > 0, cells[f"{group}_count"] / cells.population * 1000, np.nan
    )

centroids = cells.geometry.centroid
cells["x"], cells["y"] = centroids.x, centroids.y
cells["block"] = make_blocks(cells.x, cells.y, config.BLOCK_SIZE_M)

# Urban = at least half the square inside a Band A-E settlement (NISRA SDL 2015)
cov = settlement_coverage(cells, config.SETTLEMENTS_SHP)
cells["urban_share"] = cov[cov.Band.isin(config.URBAN_BANDS)].groupby("cell_id").share.sum().clip(upper=1)
cells["urban_share"] = cells["urban_share"].fillna(0)
main = cov.sort_values("share", ascending=False).drop_duplicates("cell_id").set_index("cell_id")
cells["settlement"], cells["settlement_band"] = main.Name, main.Band
cells["urban"] = cells.urban_share >= config.URBAN_MIN_SHARE
cells["study_area"] = cells.urban & (cells.population >= config.MIN_RESIDENTS)
log("grid", "urban squares (>= 50% in Band A-E settlement)", int(cells.urban.sum()))
log("grid", f"study area (urban, >= {config.MIN_RESIDENTS} residents)", int(cells.study_area.sum()))

cells["slai"] = np.nan
cells.loc[cells.study_area, "slai"] = build_slai(cells.loc[cells.study_area], config.SLAI_COMPONENTS)

# Lighting classes (Low / Medium / High tertiles of SLAI within the study area)
cells["light_class"] = "Unlit"
lit_study = cells.study_area & (cells.lit == 1)
cells.loc[lit_study, "light_class"] = pd.qcut(
    cells.loc[lit_study, "slai"].rank(method="first"), 3, labels=["Low", "Medium", "High"]
).astype(str)

cells[cells.study_area][["population", "n_lights", "slai", "light_class",
                          "street_count", "comparison_count", "n_poi"]].describe(include="all")

**Check:** almost no urban squares are unlit, so within the study area compare the
Low / Medium / High lighting classes rather than lit vs unlit.

In [ ]:
cells[cells.study_area]["light_class"].value_counts()

## 6. Save outputs

In [ ]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
cells.reset_index().to_file(config.PROCESSED_CELLS, driver="GPKG")

log_table = pd.DataFrame(cleaning_log)
log_table.to_csv(config.TABLES / "cleaning_log.csv", index=False)
print("Saved", config.PROCESSED_CELLS)
log_table